# 17.3 Packaging and Publishing

**Prerequisites:** 17.1 Environments, 17.2 pyproject.toml, 15.6 Testing in Practice, 16.5 Typing Real Code  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 `src/` layout vs flat — and the bug flat layout hides
- Building a **real wheel and sdist**, and looking inside both
- What a wheel actually is: a zip with a `dist-info/` directory
- `RECORD`, `WHEEL`, `METADATA`, `entry_points.txt` — every file explained
- 🔴 What gets included, and what silently does not
- Installing your own wheel and running its console script
- Editable installs — what `-e` really does
- `py.typed`, so your annotations reach your users (**16.5**)
- Versioning, and publishing to TestPyPI and PyPI

---

## 🔴 `src/` layout first

**15.6** made this argument for testing; it matters even more for packaging.

```
   FLAT LAYOUT                        SRC LAYOUT  (recommended)
   jobkit/                            src/
   ├── jobkit/                        └── jobkit/
   │   └── retry.py                       └── retry.py
   ├── tests/                         tests/
   └── pyproject.toml                 pyproject.toml
```

With the flat layout, `jobkit/` sits in the working directory — so `import jobkit` works
**whether or not the package is correctly installed**. Your tests pass, you publish, and a user
gets `ModuleNotFoundError` because you forgot to include a subpackage.

With `src/`, the only way to import `jobkit` is to install it. Your tests exercise the same
import path your users will.

> 🔴 **This is not a style preference.** It is the difference between testing your package and
> testing your working directory.

The project below is the one from **17.2**, plus a file deliberately left out of the
distribution.

In [ ]:
import shutil
import subprocess
import sys
import tarfile
import tempfile
import textwrap
import zipfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py173_"))
PROJECT = WORK / "jobkit"


def write(rel, source, root=None):
    path = (root or PROJECT) / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def run(args, cwd=None, timeout=900, label=None):
    done = subprocess.run(args, cwd=cwd or PROJECT, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=timeout)
    shown = label or " ".join(
        "python" if a == sys.executable else str(a) for a in args)
    body = (done.stdout + done.stderr).strip() or "(no output)"
    return (f"$ {shown}\n" + "-" * 68 + "\n" + body
            + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

In [ ]:
write("pyproject.toml", r"""
    [build-system]
    requires = ["setuptools>=68"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "jobkit"
    version = "0.1.0"
    description = "Retry policies and job scheduling helpers"
    readme = "README.md"
    requires-python = ">=3.12"
    license = "MIT"
    authors = [{name = "Aditya Tripathi"}]
    classifiers = ["Typing :: Typed"]
    dependencies = []

    [project.scripts]
    jobkit = "jobkit.cli:main"

    [tool.setuptools.packages.find]
    where = ["src"]
""")

write("README.md", "# jobkit\n\nRetry policies and job scheduling helpers.\n")
write("src/jobkit/__init__.py", '__version__ = "0.1.0"\n')
write("src/jobkit/py.typed", "")
write("src/jobkit/retry.py", r"""
    def retry_delay(attempt: int, base: float = 1.0, ceiling: float = 30.0) -> float:
        delay = base
        for _ in range(attempt):
            delay *= 2
        return min(delay, ceiling)
""")
write("src/jobkit/cli.py", r"""
    import sys

    from jobkit.retry import retry_delay


    def main() -> int:
        attempt = int(sys.argv[1]) if len(sys.argv) > 1 else 0
        print(f"attempt {attempt} -> {retry_delay(attempt):.1f}s")
        return 0
""")
write("tests/test_retry.py", r"""
    from jobkit.retry import retry_delay


    def test_ceiling() -> None:
        assert retry_delay(9) == 30.0
""")

# Deliberately NOT part of the distribution - watch where it ends up.
write("SCRATCH-NOTES.txt", "personal notes, not for users\n")
write(".env", "SECRET_TOKEN=hunter2\n")

print("project on disk:")
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print("   ", path.relative_to(PROJECT).as_posix())

## Building

```bash
pip install build
python -m build
```

`build` is a **frontend**: it reads `[build-system]` (**17.2**), creates an isolated
environment, installs the backend, and asks it for the artefacts.

🔴 **`--no-isolation` is used below** because creating that isolated environment **downloads**
the backend, which needs network access. With `setuptools` already installed, `--no-isolation`
builds with nothing but what is on the machine. In real use, leave isolation on — it is what
makes builds reproducible.

In [ ]:
built = run([sys.executable, "-m", "build", "--no-isolation"],
            label="python -m build --no-isolation").splitlines()

# setuptools is extremely chatty; keep the lines that say what happened.
print(built[0])
print("   ... (setuptools prints ~60 lines of copying) ...")
for line in built:
    if line.startswith(("adding ", "Successfully built", "* Building", "exit code")):
        print(line)

DIST = PROJECT / "dist"
print()
print("artefacts produced:")
for path in sorted(DIST.iterdir()):
    print(f"   {path.name:38} {path.stat().st_size:7,} bytes")

Two artefacts, and they are **not** interchangeable:

| | **sdist** (`.tar.gz`) | **wheel** (`.whl`) |
|---|---|---|
| Contains | source, as you wrote it | files laid out ready to copy into `site-packages` |
| Install needs | a **build step** on the user's machine | 🔴 **just unzipping** |
| Can contain C to compile | yes | no — already compiled, per platform |
| Filename encodes | name, version | name, version, **Python tag, ABI tag, platform tag** |

`jobkit-0.1.0-py3-none-any.whl` decodes as: any Python 3 (`py3`), no specific ABI (`none`), any
platform (`any`) — a pure-Python package. A compiled one would say something like
`cp312-cp312-manylinux_2_17_x86_64`.

**Publish both.** The wheel is what nearly everyone installs; the sdist is the fallback for
platforms you did not build for, and it is what lets people build from source.

## Inside a wheel

A wheel is **a zip file**. Nothing more.

In [ ]:
wheel_path = next(DIST.glob("*.whl"))

with zipfile.ZipFile(wheel_path) as wheel:
    print(f"{wheel_path.name} is a zip containing:")
    for name in sorted(wheel.namelist()):
        print("   ", name)

    print()
    print("--- WHEEL (how it was built) ---")
    print(textwrap.indent(wheel.read("jobkit-0.1.0.dist-info/WHEEL").decode(), "   "))

    print("--- entry_points.txt (the console script) ---")
    print(textwrap.indent(
        wheel.read("jobkit-0.1.0.dist-info/entry_points.txt").decode(), "   "))

    print("--- RECORD (every file, with a hash and a size) ---")
    for line in wheel.read("jobkit-0.1.0.dist-info/RECORD").decode().splitlines()[:5]:
        print("   ", line)

| File in `dist-info/` | Purpose |
|---|---|
| `METADATA` | name, version, dependencies — what `pip show` prints (**17.1**) |
| `WHEEL` | the wheel format version and what built it |
| `RECORD` | 🔴 every installed file with a **SHA-256 and size**, so uninstall is exact |
| `entry_points.txt` | the console scripts pip must generate |
| `top_level.txt` | the top-level names this package provides |

🔴 **`py.typed` is in there** — `jobkit/py.typed`. Without that file, a type checker treats your
package as untyped no matter how well annotated it is (**16.5**). It ships because it sits
inside the package directory; a marker file outside would have been silently dropped.

## Inside an sdist

In [ ]:
sdist_path = next(DIST.glob("*.tar.gz"))

with tarfile.open(sdist_path) as sdist:
    names = sorted(n for n in sdist.getnames() if not n.endswith("/"))
    print(f"{sdist_path.name} contains:")
    for name in names:
        print("   ", name)

print()
print("🔴 What is NOT in either artefact:")
in_sdist = set(names)
for unwanted in ("SCRATCH-NOTES.txt", ".env"):
    present = any(unwanted in n for n in in_sdist)
    print(f"   {unwanted:20} in sdist: {present}")

Read the two lists together.

**The sdist has `tests/` and `pyproject.toml`** — everything needed to rebuild. **The wheel does
not**: it holds only what belongs in `site-packages`.

And neither contains `SCRATCH-NOTES.txt` or `.env`. That is luck as much as design: setuptools
includes what it can identify as package data plus a standard set of metadata files.

> 🔴 **Never rely on a file being excluded by default.** A `.env`, a private key or a
> `credentials.json` inside your package directory **will** ship. Once it reaches PyPI it is
> permanently public and cannot be deleted — only yanked, after the whole internet has mirrored
> it. This curriculum's own history has a lesson about committed credentials; publishing is
> worse, because the audience is everyone.
>
> Before your first upload: **build, then list the contents**, exactly as this notebook just
> did. `MANIFEST.in` controls sdist contents when you need more than the defaults.

## Installing your own wheel

The real test of a package is installing it somewhere clean and using it. Below: a throwaway
virtual environment (**17.1**), the wheel installed from a local file, then both the import and
the **generated console script**.

In [ ]:
ENV = WORK / "consumer-venv"
print(run([sys.executable, "-m", "venv", str(ENV)], timeout=300,
          label="python -m venv consumer-venv"))

bindir = ENV / ("Scripts" if sys.platform == "win32" else "bin")
env_python = bindir / ("python.exe" if sys.platform == "win32" else "python")

print()
print(run([str(env_python), "-m", "pip", "install", "--quiet", str(wheel_path)],
          timeout=600, label=f"consumer-venv/python -m pip install {wheel_path.name}"))

In [ ]:
print(run([str(env_python), "-c",
           "import jobkit;"
           "from jobkit.retry import retry_delay;"
           "print('version   :', jobkit.__version__);"
           "print('retry(5)  :', retry_delay(5));"
           "print('located at:', jobkit.__file__)"],
          cwd=WORK, label="consumer-venv/python -c 'import jobkit ...'"))

print()
script = bindir / ("jobkit.exe" if sys.platform == "win32" else "jobkit")
print("the console script pip generated:", script.name, "->", script.exists())
if script.exists():
    print(run([str(script), "4"], cwd=WORK, label="jobkit 4"))

The package imported from **`site-packages`** of a different environment, and
`jobkit` became a real command — generated by pip from the `entry_points.txt` you saw in the
wheel.

That round trip is the only proof that packaging worked.

## Editable installs

```bash
pip install -e .
```

An **editable** install puts a pointer to your source directory into `site-packages` instead of
copying files. Edit the source and the change is live — no reinstall.

| | Regular install | Editable (`-e`) |
|---|---|---|
| Files in `site-packages` | copies of your code | a `.pth` file pointing at your source |
| Edit source, see change | ❌ reinstall | ✅ immediately |
| Use for | consumers, CI, production | 🔴 **local development, always** |

🔴 It requires the build backend to be available. In a fresh 3.12+ venv, `setuptools` is **not
installed**, so `pip install -e . --no-build-isolation` fails with a confusing error — the fix
is either to allow isolation (needs network) or `pip install setuptools` first.

> `pip install -e ".[dev]"` (**17.2**) is the single command that sets up a project for work:
> your package importable, plus every development tool.

## Versioning

`0.1.0` is **SemVer**: `MAJOR.MINOR.PATCH`.

| Bump | When | Example |
|---|---|---|
| **PATCH** `0.1.0 → 0.1.1` | a bug fix, no API change | fixed the ceiling calculation |
| **MINOR** `0.1.0 → 0.2.0` | new functionality, backwards compatible | added `jitter=` |
| **MAJOR** `0.1.0 → 1.0.0` | 🔴 a **breaking** change | renamed or removed something public |

Two conventions worth knowing:

- **`0.x` means "no promises"** — under 1.0.0, anything may break. Publishing `1.0.0` is a
  commitment.
- 🔴 **A version can never be reused on PyPI.** Upload `0.1.0`, spot a mistake, and your only
  options are yanking it and publishing `0.1.1`. This is why you test with TestPyPI first.

## Publishing

**Not executed here** — this notebook will not upload anything. The commands, in order:

```bash
python -m build                       # 1. build both artefacts
python -m twine check dist/*          # 2. validate the metadata locally
python -m twine upload \
    --repository testpypi dist/*      # 3. TestPyPI first, always
pip install --index-url \
    https://test.pypi.org/simple/ jobkit   # 4. install it from there and try it
python -m twine upload dist/*         # 5. only now, the real thing
```

🔴 **Step 3 is not optional.** TestPyPI is a full copy of PyPI that nobody depends on, and it is
the only place you can make a packaging mistake safely.

### Credentials

**Do not put a PyPI password in a file.** Two better options:

| Method | How |
|---|---|
| **API token** | generate on PyPI, scope it to one project, store in `~/.pypirc` or `TWINE_PASSWORD` |
| 🔴 **Trusted publishing** | GitHub Actions authenticates to PyPI with OIDC — **no token exists at all** |

Trusted publishing is the current recommendation: nothing to leak, nothing to rotate.

```yaml
      - uses: pypa/gh-action-pypi-publish@release/v1     # no password anywhere
```

> Secrets and configuration generally — `.env` files, `os.environ`, secret managers — are
> **17.2**'s neighbour topic and come up again in **18 Working with APIs**, where the tokens are
> real.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Flat layout.** `import jobkit` works from the project root whether or not the package is installed correctly, so packaging bugs reach your users (**15.6**).
2. 🔴 **Publishing without listing the artefact contents first.** A `.env` or key inside your package directory ships, and PyPI uploads cannot be deleted.
3. **Uploading straight to PyPI.** TestPyPI exists precisely so your first mistake is free.
4. **Expecting to reuse a version number.** Once `0.1.0` is on PyPI it is gone forever; you can only yank and bump.
5. **Forgetting `py.typed`.** Your users' type checkers ignore every annotation you wrote (**16.5**).
6. **Publishing only an sdist.** Everyone then needs a build backend to install your package; a wheel just unzips.
7. **Storing a PyPI password in a file.** Use a scoped API token, or trusted publishing and have no credential at all.
8. **Running `pip install -e .` in a fresh 3.12+ venv with `--no-build-isolation`.** `setuptools` is not there by default and the error does not say so clearly.
9. **Treating `0.x` as stable.** Under 1.0.0 you have promised nothing — and neither has anything you depend on.

## Best Practices

- Use the `src/` layout, so tests import the way users will.
- Build both a wheel and an sdist, and publish both.
- 🔴 Unzip your own wheel and read the file list before every first upload.
- Ship `py.typed` inside the package directory if your code is annotated.
- Use `pip install -e '.[dev]'` for local work — one command sets everything up.
- Test the artefact by installing it into a clean venv and running it.
- Go through TestPyPI before PyPI, every time.
- Prefer trusted publishing to API tokens, and scoped tokens to account-wide ones.
- Follow SemVer, and mean it once you publish 1.0.0.

## Practice Exercises

Try these before moving on.

1. Take a module you have written, give it a `pyproject.toml` and a `src/` layout, and build a wheel. How many files are in it?
2. 🔴 Unzip your wheel and read `RECORD`. What would `pip uninstall` remove, and how does it know the files have not changed?
3. Add a `SECRET.txt` inside your package directory, rebuild, and find it in the wheel. Now move it outside and confirm it is gone. Which was luck and which was design?
4. Install your wheel into a fresh venv and run it. Then delete the source directory and confirm it still works — that proves nothing was importing from your working copy.
5. Add a `[project.scripts]` entry, reinstall, and find the generated executable in the consumer venv's `Scripts/` or `bin/`.
6. Compare a flat-layout and a `src`-layout copy of the same project: deliberately omit a subpackage from `packages.find` and see which layout catches it.
7. 🔴 Publish something to **TestPyPI** and install it from there. What did you get wrong the first time? (Almost everyone gets something wrong the first time.)
8. **Interview question:** what is the difference between a wheel and an sdist, and why would a project publish both?

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | 🔴 `venv` no longer pre-installs `setuptools`, which is why `--no-build-isolation` fails in a fresh environment |
| **PEP 660 (2021)** | editable installs standardised, so `pip install -e .` works with any backend |
| **PEP 621 (2020)** | `[project]` metadata — the same table every backend reads (**17.2**) |
| **PEP 517/518** | build frontends and pluggable backends, which is what `python -m build` implements |
| **PEP 427** | the wheel format — a zip plus `dist-info/` |

## Where next

| Notebook | Covers |
|---|---|
| **17.4** | `ruff` — linting and formatting |
| **17.5** | profiling and performance |

## Related

- **17.1 Environments** — venvs, `site-packages` and where an install lands
- **17.2 pyproject.toml** — the metadata this notebook builds from
- **15.6 Testing in Practice** — the `src/` layout argument, from the testing side
- **16.5 Typing Real Code** — `py.typed`, and why it must be inside the package
- **18 Working with APIs** — where real tokens and secret handling turn up